In [2]:
%pip install lxml html5lib beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [html5lib]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import lxml
import html5lib
import bs4

print("Tout est installé !")

Tout est installé !


In [4]:
import pandas as pd

url = "https://fgeerolf.com/data/notaires/HistoriquedesprixaumappartementsanciensParispararrdt.html"

tables = pd.read_html(url)

print("Nombre de tableaux :", len(tables))

for i, table in enumerate(tables):
    print(f"\nTableau {i}")
    print(table.head())

ValueError: No tables found

In [6]:
%pip install pdfplumber

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 27.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 31.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pdfplumber]3 [pdfminer.six]
Note: you may need to restart the kernel to use updated packages.


In [7]:
import requests
import pdfplumber

url = "https://paris.notaires.fr/sites/default/files/HistoriquedesprixaumappartementsanciensParispararrdt_2.pdf"

r = requests.get(url)
r.raise_for_status()

with open("prix_paris.pdf", "wb") as f:
    f.write(r.content)

with pdfplumber.open("prix_paris.pdf") as pdf:
    for i, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        if tables:
            print("Page :", i + 1)
            print(tables[0][:5])
            break

Page : 1
[['Trimestre', 'Centre', '1er', '2e', '3e', '4e', '5e', '6e', '7e', '8e', '9e', '10e', '11e', '12e', '13e', '14e', '15e', '16e', '17e', '18e', '19e', '20e'], ['T4 2025', '11 490 €', '12 100 €', '11 180 €', '12 540 €', '13 020 €', '11 620 €', '13 400 €', '12 940 €', '11 500 €', '10 280 €', '8 900 €', '9 640 €', '8 700 €', '8 510 €', '9 350 €', '8 860 €', '10 050 €', '9 360 €', '8 530 €', '7 560 €', '8 300 €'], ['T3 2025', '11 240 €', '12 020 €', '11 310 €', '12 340 €', '12 750 €', '11 410 €', '14 010 €', '13 180 €', '12 190 €', '10 340 €', '9 040 €', '9 560 €', '8 700 €', '8 720 €', '9 240 €', '8 960 €', '10 250 €', '9 340 €', '8 550 €', '7 850 €', '8 290 €'], ['T2 2025', '11 370 €', '11 830 €', '10 870 €', '12 760 €', '12 680 €', '11 350 €', '13 310 €', '12 830 €', '11 710 €', '10 110 €', '8 870 €', '9 290 €', '8 500 €', '8 500 €', '9 060 €', '8 880 €', '10 040 €', '9 280 €', '8 410 €', '7 430 €', '8 070 €'], ['T1 2025', '11 540 €', '12 740 €', '11 400 €', '12 280 €', '13 030 

In [9]:
print(df_paris.columns.tolist())

['Trimestre', 'Centre', '1er', '2e', '3e', '4e', '5e', '6e', '7e', '8e', '9e', '10e', '11e', '12e', '13e', '14e', '15e', '16e', '17e', '18e', '19e', '20e', 'annee']


In [13]:
import pandas as pd
import numpy as np
import pdfplumber
import re

all_tables = []

with pdfplumber.open("prix_paris.pdf") as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()

        for table in tables:
            if table and "Trimestre" in table[0]:
                df_temp = pd.DataFrame(table[1:], columns=table[0])
                all_tables.append(df_temp)

df_paris = pd.concat(all_tables, ignore_index=True)

# Garder uniquement les lignes correspondant aux trimestres
df_paris = df_paris[
    df_paris["Trimestre"].astype(str).str.match(r"T[1-4] \d{4}")
].copy()

# Extraire l'année
df_paris["annee"] = (
    df_paris["Trimestre"].str.extract(r"(\d{4})")[0].astype(int)
)

# Garder le T4 de chaque année
df_paris = df_paris[df_paris["Trimestre"].str.startswith("T4")].copy()

# Colonnes des arrondissements
arrondissements = ["1er"] + [f"{i}e" for i in range(2, 21)]

# Nettoyer les valeurs numériques
for col in arrondissements:
    df_paris[col] = pd.to_numeric(
        df_paris[col].astype(str).str.replace(r"[^\d]", "", regex=True),
        errors="coerce"
    )

# Construire le DataFrame avec les codes INSEE en index
paris_prix = df_paris[["annee"] + arrondissements].copy()

paris_prix = paris_prix.rename(
    columns={f"{i}e": f"75{i:03d}" for i in range(1, 21)}
)

paris_prix = paris_prix.set_index("annee").T
paris_prix.index.name = "code_insee"
paris_prix.columns = paris_prix.columns.astype(int)
paris_prix.reindex()
display(paris_prix)

annee,2025,2024,2023,2022,2021,2020,2019,2018,2017,2016,...,2004,2003,2002,2001,2000,1999,1998,1997,1996,1995
code_insee,,,,,,,,,,,,,,,,,,,,,
1er,12100,12180,13550,14770,13650,13250,13400,12050,11560,10250,...,6510,5100,5010,3940,3500,3370,2900,2880,2960,3760
75002,11180,11580,11830,12100,11830,12620,11760,11050,10890,9630,...,5240,4610,3840,3440,3020,2580,2490,2280,2320,2220
75003,12540,12040,12570,13120,12900,12940,12790,12250,11170,10260,...,5380,4810,3970,3610,3290,3040,2520,2510,2510,2670
75004,13020,12390,13080,13110,12720,13710,13670,13360,11540,11310,...,6560,5550,4950,4320,4110,3560,3440,3130,2960,3170
75005,11620,11070,12000,12710,12650,13480,12590,11600,11100,10720,...,6220,5450,4750,4470,4090,3580,3270,3140,3060,3240
75006,13400,13460,13840,14350,13960,13810,13150,12770,12360,11240,...,6660,6110,5860,5400,5050,4490,3680,3570,3370,3860
75007,12940,12420,13190,13800,13290,13010,12980,12280,11530,11130,...,6330,5770,5380,4940,4720,4190,3700,3300,3250,3830
75008,11500,11760,11230,11550,11920,11620,10830,10500,9900,9210,...,5530,5110,4670,4270,3910,3380,3040,2810,3230,3520
75009,10280,9650,10120,10960,10780,11260,10790,9710,9560,8600,...,4330,3760,3460,3170,2880,2620,2280,2120,2160,2200


In [17]:

import pandas as pd
import numpy as np
import requests
import pdfplumber
import re
from io import BytesIO

url = "https://paris.notaires.fr/sites/default/files/HistoriquedesprixaumappartementsanciensParispararrdt_2.pdf"
response = requests.get(url)
response.raise_for_status()

all_tables = []

with pdfplumber.open(BytesIO(response.content)) as pdf:
    for page in pdf.pages:
        for table in page.extract_tables():
            if table and "Trimestre" in table[0]:
                all_tables.append(
                    pd.DataFrame(table[1:], columns=table[0])
                )

df_paris = pd.concat(all_tables, ignore_index=True)

df_paris.columns = df_paris.columns.astype(str).str.strip()

df_paris = df_paris[
    df_paris["Trimestre"].astype(str).str.match(r"T[1-4] \d{4}")
].copy()

df_paris["annee"] = df_paris["Trimestre"].str.extract(r"(\d{4})").astype(int)

df_paris = df_paris[
    df_paris["Trimestre"].str.startswith("T4")
].copy()

arrondissements = ["1er"] + [f"{i}e" for i in range(2, 21)]

for col in arrondissements:
    df_paris[col] = pd.to_numeric(
        df_paris[col].astype(str).str.replace(r"[^\d]", "", regex=True),
        errors="coerce"
    )

paris_prix = df_paris[["annee"] + arrondissements].copy()

paris_prix = paris_prix.rename(
    columns={col: f"75{int(re.search(r'\d+', col).group()):03d}"
             for col in arrondissements}
)

paris_prix = paris_prix.set_index("annee").T
paris_prix.index.name = "code_insee"
paris_prix = paris_prix.reset_index()

annees = sorted(
    col for col in paris_prix.columns
    if col != "code_insee" and 2014 <= col <= 2025
)

paris_prix = paris_prix[["code_insee"] + annees]
paris_prix["code_insee"] = paris_prix["code_insee"].astype(str)


paris_prix.head()

annee,code_insee,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,75001,10690,11030,10250,11560,12050,13400,13250,13650,14770,13550,12180,12100
1,75002,9380,9290,9630,10890,11050,11760,12620,11830,12100,11830,11580,11180
2,75003,9620,9550,10260,11170,12250,12790,12940,12900,13120,12570,12040,12540
3,75004,10730,11130,11310,11540,13360,13670,13710,12720,13110,13080,12390,13020
4,75005,10540,10300,10720,11100,11600,12590,13480,12650,12710,12000,11070,11620


In [ ]:
# 7. Fusionner dans Prix_immo : remplacer uniquement les NaN
Prix_immo["code_insee"] = Prix_immo["code_insee"].astype(str).str.zfill(5)

paris_prix = paris_prix.set_index("code_insee")

for year in annees:
    mask = Prix_immo["code_insee"].isin(paris_prix.index)
    Prix_immo.loc[mask, year] = Prix_immo.loc[mask, year].fillna(
        Prix_immo.loc[mask, "code_insee"].map(paris_prix[year])
    )

print("Fusion terminée !")
print("Nombre d'arrondissements :", len(paris_prix))
display(Prix_immo[Prix_immo["code_insee"].isin(paris_prix.index)])